# QLoRA Fine-Tuning v2: Qwen2.5-1.5B-Instruct for SEO & GEO Pipeline
**Project:** AI SEO & GEO Blog Builder (Module 1 — Research & Content Brief) — **Model v2 (Full UI-Coverage)**
**Hardware:** Google Colab Free Tier (1x NVIDIA T4 GPU - 15GB VRAM)

## Nâng cấp cốt lõi v2 (100% dữ liệu sạch & phủ kín mọi trường hợp UI)
- **2,400 mẫu song ngữ sạch 100%** (50% EN / 50% VI), không gộp data lỗi từ v1
- **Task 1 (`KEYWORD_RESEARCH`, 1,200 mẫu):** 75% 1 topic + 25% nhiều topics; đủ cả 4 tổ hợp Business/Target; 4 nhãn Intent cân bằng
- **Task 2 (`CONTENT_BRIEF`, 1,200 mẫu):** 45% 1 từ có `Known intent` + 40% Bulk chọn 2–5 từ khóa + 15% từ khóa tự do; 80% có Niche / 20% trống Niche; đủ cả 4 tổ hợp Business/Target
- **Đồng bộ 100% Prompt** giữa Train (`generate_dataset.py`) và Inference (`serve.py`)

**Hướng dẫn:** Upload `train_v2.jsonl` và `test_v2.jsonl` vào `/content/` rồi bấm **Runtime → Run All**.

## 1. Environment & GPU Check

In [ ]:
!nvidia-smi

In [ ]:
!pip uninstall -y torchaudio
!pip install -q -U transformers datasets trl peft bitsandbytes accelerate

## 2. Load 4-Bit Base Model & Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

# Ensure all non-quantized weights and buffers are strictly float16 (no bfloat16 on T4 GPU)
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
for buffer in model.buffers():
    if buffer.dtype == torch.bfloat16:
        buffer.data = buffer.data.to(torch.float16)

## 3. Configure LoRA (PEFT)
Target all linear projections: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Ensure all LoRA trainable weights are float32 so GradScaler works reliably on T4 GPU
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.print_trainable_parameters()

## 4. Load Dataset & Format ChatML
**Instruction:** Drag and drop `train.jsonl` and `test.jsonl` from your computer into the Files pane on the left (`/content/`).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "train_v2.jsonl",
    "test": "test_v2.jsonl"
})

def apply_chat_template(sample):
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

formatted_dataset = dataset.map(apply_chat_template)
print("Sample formatted prompt:\n", formatted_dataset["train"][0]["text"][:450])

## 5. Training with SFTTrainer (3 Epochs)

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./outputs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=1024,
    dataset_text_field="text",
    packing=False,
    fp16=False,
    bf16=False,
    report_to="none"
)

# Ensure all LoRA trainable parameters are float32 and NO bfloat16 exists on T4 GPU
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
    elif param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
for buffer in model.buffers():
    if buffer.dtype == torch.bfloat16:
        buffer.data = buffer.data.to(torch.float16)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
    processing_class=tokenizer,
    args=training_args
)

trainer.train()

## 6. Save Adapter & Download

In [ ]:
ADAPTER_DIR = "./qwen2.5_1.5b_qlora_adapter"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

!zip -r qwen2.5_1.5b_qlora_adapter.zip ./qwen2.5_1.5b_qlora_adapter

from google.colab import files
files.download("qwen2.5_1.5b_qlora_adapter.zip")
print("[SUCCESS] Adapter packaging complete. Download started!")

## 7. Direct Inference Test

In [ ]:
system_kw = """You are an expert SEO & GEO (Generative Engine Optimization) analyst for Shopify e-commerce blogs.
Given a niche or topic, language, and optional business context, return ONLY a JSON object (no markdown, no explanation) in this exact schema:
{
  "keywords": [
    { "keyword": "<string>", "intent": "Informational | Commercial | Transactional | Navigational" }
  ]
}
Rules:
- Return 6-9 keyword objects covering diverse search intents.
- keyword: 2-7 words, natural search query style with accurate spelling and full diacritics.
- intent must strictly follow these rules:
  * Transactional: explicit buying, ordering, pricing, or deal intent (buy, order, price, deal, discount, mua, bán, giá, đặt mua, khuyến mãi, báo giá).
  * Commercial: comparing, reviewing, or researching products before buying (best, top, review, vs, compare, đánh giá, so sánh, tốt nhất, nên mua, loại nào tốt, product + audience/budget).
  * Informational: learning, how-to guides, tips, definitions, troubleshooting (how to, what is, why, tips, guide, cách, hướng dẫn, là gì, tại sao, kinh nghiệm, bảo quản).
  * Navigational: searching for a specific brand, official website, or store location (official store, website, login, cửa hàng, đại lý chính hãng, trang chủ + brand name).
- No volume, KD, or CPC fields — only keyword + intent.
- Language of keywords must strictly match the Language parameter."""

system_cb = """You are an expert SEO & GEO content strategist for Shopify e-commerce blogs.
Given a primary keyword (or a list of selected keywords) and optional context, return ONLY a JSON object (no markdown, no explanation) in this exact schema:
{
  "primary_keyword": "<string>",
  "secondary_keywords": ["<string>", ...],
  "dominant_intent": "Informational | Commercial | Transactional | Navigational",
  "funnel_stage": "TOFU | MOFU | BOFU",
  "recommended_content_type": "Informational | Buyer's guide | How-to | FAQ",
  "recommended_style": "Informative | Analytical | Instructive | Persuasive | Conversational",
  "recommended_tone": "Authoritative | Caring | Friendly | Informative | Formal | Conversational",
  "target_audience": "<string>",
  "key_talking_points": ["<string with E-E-A-T: stat or credential>", ...],
  "geo_blueprint": {
    "answer_block_focus": "<string>",
    "comparison_table_idea": "<string>",
    "faq_questions": ["<question>", "<question>"]
  },
  "suggested_cta": "<string>"
}
Rules:
- If Selected keywords are provided: pick the broadest, highest-value topic as primary_keyword, put ALL remaining selected keywords into secondary_keywords (optionally add 1 more if needed so there are 2-5 items), and synthesize a unified dominant_intent and strategy covering every selected keyword.
- If a single Primary keyword is provided: keep it as primary_keyword, generate 2-4 semantically related secondary_keywords, and if Known intent is provided, dominant_intent must match it.
- key_talking_points: 3-5 items, each containing an E-E-A-T signal (stat, study, certification, or expert benchmark).
- faq_questions: 2-4 conversational AI-search questions.
- Intent-to-Funnel alignment: Informational->TOFU, Commercial->MOFU, Transactional->BOFU, Navigational->MOFU or BOFU.
- All enum fields must use exactly one listed value.
- Language of all text fields must strictly match the Language parameter with accurate spelling and diacritics."""

def quick_infer(sys_p, usr_p, max_tok=420):
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": usr_p}],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tok, temperature=0.2, top_p=0.85, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== TEST 1: KEYWORD RESEARCH (VI) ===")
print(quick_infer(system_kw, "Niche: Ghế công thái học\nLanguage: Vietnamese\nGenerate a keyword list for this niche.", 320))

print("\n=== TEST 2: SINGLE CONTENT BRIEF (VI) ===")
print(quick_infer(system_cb, "Primary keyword: mua ghế công thái học chính hãng\nNiche: Ghế công thái học\nLanguage: Vietnamese\nKnown intent: Transactional\nGenerate a content brief.", 550))

print("\n=== TEST 3: BULK MULTI-KEYWORD BRIEF (VI) ===")
print(quick_infer(system_cb, "Selected keywords:\n- ghế công thái học loại nào tốt (Commercial)\n- cách chỉnh ghế công thái học đúng tư thế (Informational)\n- mua ghế công thái học giá dưới 5 triệu (Transactional)\nNiche: Ghế công thái học\nLanguage: Vietnamese\nGenerate a unified content brief covering all selected keywords.", 550))